# timesfm experiment v4 — lora adaptation + capped xreg hybrid

this experiment is the first walmart-specific adaptation of timesfm. the 231m pretrained base parameters stay frozen and small lora adapters are trained on leakage-safe historical store–dept windows. the resulting lora forecast is combined with the already validated v3 xreg forecast, seasonal-naive, raw and residual candidates. xreg is capped at 10% because v3.1 showed real but regime-dependent value.

In [ ]:
# keep colab's pinned numpy/pandas; install only model and tracking dependencies.
%pip install -q "transformers==5.13.1" "peft>=0.18,<0.19" "accelerate>=1.12,<2" "wandb==0.28.0"

In [ ]:
import gc, hashlib, json, math, platform, random, shutil, time, warnings
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch, wandb
from peft import LoraConfig, PeftModel, get_peft_model
from torch.utils.data import DataLoader, Dataset
from transformers import TimesFm2_5ModelForPrediction
warnings.filterwarnings('ignore')
torch.set_float32_matmul_precision('high')
print({'python':platform.python_version(),'torch':torch.__version__,'pandas':pd.__version__,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'})

In [ ]:
config={
 'data_dir':'/content/drive/MyDrive/walmart_competition_data',
 'output_dir':'/content/artifacts/timesfm_v4_lora_xreg',
 'model_id':'google/timesfm-2.5-200m-transformers',
 'v3_artifact':'timesfm-v3-xreg-corrected-calibration:latest',
 'validation_weeks':39,'blend_calibration_weeks':20,'lora_validation_weeks':13,
 'context_len':32,'train_horizon':13,'num_train_samples':6000,
 'epochs':6,'batch_size':8,'gradient_accumulation':4,'learning_rate':5e-5,
 'weight_decay':0.01,'max_grad_norm':1.0,'early_stopping_patience':2,
 'lora_r':4,'lora_alpha':8,'lora_dropout':0.05,
 'inference_batch_size':32,'weight_step':0.05,'xreg_weight_cap':0.10,
 'holiday_weight':5.0,'clip_min':0.0,'clip_max':300000.0,'seed':42,
 'wandb_entity':'kende23-n-a','wandb_project':'Walmart-Recruiting---Store-Sales-Forecasting',
 'wandb_run_name':'timesfm_v4_lora_xreg_hybrid',
}
data_dir,output_dir=Path(config['data_dir']),Path(config['output_dir']);adapter_dir=output_dir/'best_lora_adapter';output_dir.mkdir(parents=True,exist_ok=True)
random.seed(config['seed']);np.random.seed(config['seed']);torch.manual_seed(config['seed'])
if torch.cuda.is_available():torch.cuda.manual_seed_all(config['seed'])
print(config)

In [ ]:
try:
 from google.colab import drive,userdata
 drive.mount('/content/drive')
 wandb_key=userdata.get('WANDB_API_KEY')
except Exception:
 wandb_key=None
wandb.login(key=wandb_key) if wandb_key else wandb.login()

## data matrix and protected chronological splits

In [ ]:
def locate_csv(root,name):
 for path in [root/name,root/f'{name}.zip']:
  if path.exists():return path
 raise FileNotFoundError(f'missing {name} or {name}.zip in {root}')
def wmae(y,p,h):
 y,p=np.asarray(y,float),np.asarray(p,float);weights=np.where(np.asarray(h,bool),config['holiday_weight'],1.0)
 return float(np.sum(weights*np.abs(y-p))/np.sum(weights))
train=pd.read_csv(locate_csv(data_dir,'train.csv'),parse_dates=['Date']).sort_values(['Store','Dept','Date']).reset_index(drop=True)
train['Weekly_Sales']=pd.to_numeric(train.Weekly_Sales,errors='coerce').fillna(0).astype('float32');train['IsHoliday']=train.IsHoliday.astype(bool)
all_dates=pd.DatetimeIndex(sorted(train.Date.unique()));final_start=len(all_dates)-config['validation_weeks'];blend_cal_start=final_start-config['blend_calibration_weeks'];lora_val_start=blend_cal_start-config['lora_validation_weeks']
all_keys=sorted(map(tuple,train[['Store','Dept']].drop_duplicates().to_numpy()));key_to_all={k:i for i,k in enumerate(all_keys)}
lookup=train.set_index(['Store','Dept','Date']).Weekly_Sales;sales=np.zeros((len(all_keys),len(all_dates)),np.float32);observed=np.zeros_like(sales,dtype=bool)
for i,(store,dept) in enumerate(all_keys):
 idx=pd.MultiIndex.from_product([[store],[dept],all_dates],names=['Store','Dept','Date']);values=lookup.reindex(idx);observed[i]=values.notna().to_numpy();sales[i]=values.fillna(0).to_numpy(np.float32)
holiday_by_date=train.groupby('Date').IsHoliday.max().reindex(all_dates).fillna(False).to_numpy(bool)
print({'all_rows':len(train),'all_series':len(all_keys),'lora_train_end':str(all_dates[lora_val_start-1].date()),'lora_validation':f'{all_dates[lora_val_start].date()} -> {all_dates[blend_cal_start-1].date()}','blend_calibration':f'{all_dates[blend_cal_start].date()} -> {all_dates[final_start-1].date()}','final_validation':f'{all_dates[final_start].date()} -> {all_dates[-1].date()}'})

## random-window lora dataset

training targets end before the lora-validation period. missing store–dept observations receive zero loss weight rather than being treated as real zero sales. holiday target weeks receive weight 5.

In [ ]:
class random_window_dataset(Dataset):
 def __init__(self,values,mask,holidays,end_pos,context,horizon,num_samples,seed):
  self.values,self.mask,self.holidays=values,mask,holidays;self.context,self.horizon=context,horizon;self.samples=[]
  rng=np.random.default_rng(seed);max_start=end_pos-context-horizon
  if max_start<0:raise ValueError('not enough history for lora windows')
  valid=[]
  for series_idx in range(len(values)):
   starts=[s for s in range(max_start+1) if mask[series_idx,s+context:s+context+horizon].any()]
   if starts:valid.append((series_idx,starts))
  for _ in range(num_samples):
   series_idx,starts=valid[rng.integers(len(valid))];self.samples.append((series_idx,int(starts[rng.integers(len(starts))])))
 def __len__(self):return len(self.samples)
 def __getitem__(self,index):
  i,start=self.samples[index];cut=start+self.context;target_slice=slice(cut,cut+self.horizon)
  context=torch.tensor(self.values[i,start:cut],dtype=torch.float32);target=torch.tensor(self.values[i,target_slice],dtype=torch.float32)
  weights=np.where(self.holidays[target_slice],config['holiday_weight'],1.0)*self.mask[i,target_slice];weights=torch.tensor(weights,dtype=torch.float32)
  return context,target,weights
class fixed_validation_dataset(Dataset):
 def __init__(self,values,mask,holidays,start,context,horizon):
  self.items=[]
  for i in range(len(values)):
   target_slice=slice(start,start+horizon);weights=np.where(holidays[target_slice],config['holiday_weight'],1.0)*mask[i,target_slice]
   if weights.sum()>0:self.items.append((torch.tensor(values[i,start-context:start],dtype=torch.float32),torch.tensor(values[i,target_slice],dtype=torch.float32),torch.tensor(weights,dtype=torch.float32)))
 def __len__(self):return len(self.items)
 def __getitem__(self,index):return self.items[index]
train_ds=random_window_dataset(sales,observed,holiday_by_date,lora_val_start,config['context_len'],config['train_horizon'],config['num_train_samples'],config['seed'])
val_ds=fixed_validation_dataset(sales,observed,holiday_by_date,lora_val_start,config['context_len'],config['lora_validation_weeks'])
train_loader=DataLoader(train_ds,batch_size=config['batch_size'],shuffle=True,num_workers=0,pin_memory=True);val_loader=DataLoader(val_ds,batch_size=config['inference_batch_size'],shuffle=False,num_workers=0,pin_memory=True)
print({'train_samples':len(train_ds),'train_batches':len(train_loader),'validation_series':len(val_ds),'validation_batches':len(val_loader)})

## load timesfm, attach lora, and start the tracked training run

In [ ]:
if not torch.cuda.is_available():raise RuntimeError('enable a t4 gpu before lora training')
device=torch.device('cuda');base_model=TimesFm2_5ModelForPrediction.from_pretrained(config['model_id'],torch_dtype=torch.float16,device_map='cuda')
lora_cfg=LoraConfig(r=config['lora_r'],lora_alpha=config['lora_alpha'],target_modules='all-linear',lora_dropout=config['lora_dropout'],bias='none')
model=get_peft_model(base_model,lora_cfg);trainable=sum(p.numel() for p in model.parameters() if p.requires_grad);total=sum(p.numel() for p in model.parameters())
model.print_trainable_parameters();print({'trainable':trainable,'total':total,'trainable_pct':100*trainable/total})
run=wandb.init(entity=config['wandb_entity'],project=config['wandb_project'],group='timesfm-experiments',job_type='lora_finetuning',name=config['wandb_run_name'],config=config)
run.summary.update({'model/base_parameters':total-trainable,'model/trainable_lora_parameters':trainable,'model/trainable_pct':100*trainable/total})

In [ ]:
optimizer=torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),lr=config['learning_rate'],weight_decay=config['weight_decay'])
updates_per_epoch=math.ceil(len(train_loader)/config['gradient_accumulation']);scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max(1,config['epochs']*updates_per_epoch))
scaler=torch.amp.GradScaler('cuda');best_val=float('inf');patience=0;history=[];training_started=time.time()
if adapter_dir.exists():shutil.rmtree(adapter_dir)
for epoch in range(1,config['epochs']+1):
 model.train();optimizer.zero_grad(set_to_none=True);train_num=train_den=0.0
 for step,(context,target,weights) in enumerate(train_loader,1):
  context,target,weights=context.to(device,non_blocking=True),target.to(device,non_blocking=True),weights.to(device,non_blocking=True)
  with torch.autocast('cuda',dtype=torch.float16):
   output=model(past_values=context,forecast_context_len=config['context_len'],return_dict=True);prediction=output.mean_predictions[:,:config['train_horizon']]
   numerator=(weights*torch.abs(prediction.float()-target)).sum();denominator=weights.sum().clamp_min(1.0);loss=(numerator/denominator)/config['gradient_accumulation']
  scaler.scale(loss).backward();train_num+=float(numerator.detach().cpu());train_den+=float(denominator.detach().cpu())
  if step%config['gradient_accumulation']==0 or step==len(train_loader):
   scaler.unscale_(optimizer);torch.nn.utils.clip_grad_norm_(model.parameters(),config['max_grad_norm']);scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True);scheduler.step()
  if step%100==0:print({'epoch':epoch,'step':step,'batches':len(train_loader),'elapsed_min':round((time.time()-training_started)/60,2)})
 model.eval();val_num=val_den=0.0
 with torch.inference_mode():
  for context,target,weights in val_loader:
   context,target,weights=context.to(device),target.to(device),weights.to(device)
   with torch.autocast('cuda',dtype=torch.float16):prediction=model(past_values=context,forecast_context_len=config['context_len'],return_dict=True).mean_predictions[:,:config['lora_validation_weeks']]
   val_num+=float((weights*torch.abs(prediction.float()-target)).sum().cpu());val_den+=float(weights.sum().cpu())
 train_wmae=train_num/max(train_den,1);val_wmae=val_num/max(val_den,1);row={'epoch':epoch,'train_wmae':train_wmae,'lora_validation_wmae':val_wmae,'lr':scheduler.get_last_lr()[0],'elapsed_minutes':(time.time()-training_started)/60};history.append(row);run.log(row,step=epoch);print(row)
 if val_wmae<best_val-1e-6:
  best_val=val_wmae;patience=0;model.save_pretrained(adapter_dir);run.summary.update({'training/best_epoch':epoch,'training/best_lora_validation_wmae':best_val})
 else:
  patience+=1
  if patience>=config['early_stopping_patience']:
   print({'early_stopping_epoch':epoch,'best_val_wmae':best_val});break
training_minutes=(time.time()-training_started)/60;history_df=pd.DataFrame(history);print({'training_minutes':training_minutes,'best_val_wmae':best_val,'best_adapter':str(adapter_dir)})

## reload the best adapter and forecast calibration/final horizons

In [ ]:
del model,base_model,optimizer,scheduler,scaler;gc.collect();torch.cuda.empty_cache()
base_model=TimesFm2_5ModelForPrediction.from_pretrained(config['model_id'],torch_dtype=torch.float16,device_map='cuda');ft_model=PeftModel.from_pretrained(base_model,adapter_dir).eval()
def forecast_lora(matrix,horizon,label):
 started=time.time();parts=[]
 with torch.inference_mode():
  for start in range(0,len(matrix),config['inference_batch_size']):
   batch=torch.as_tensor(matrix[start:start+config['inference_batch_size']],dtype=torch.float16,device=device)
   with torch.autocast('cuda',dtype=torch.float16):pred=ft_model(past_values=batch,return_dict=True).mean_predictions[:,:horizon]
   parts.append(pred.float().cpu().numpy())
 result=np.clip(np.nan_to_num(np.concatenate(parts),nan=0,posinf=config['clip_max'],neginf=0),config['clip_min'],config['clip_max']);print({'label':label,'shape':result.shape,'minutes':(time.time()-started)/60});return result
cal_lora=forecast_lora(sales[:,:blend_cal_start],config['blend_calibration_weeks'],'lora calibration');final_lora=forecast_lora(sales[:,:final_start],config['validation_weeks'],'lora final')

## reuse v3 xreg and baseline candidates

the official xreg wrapper cannot directly ingest a peft adapter. therefore the hybrid uses the exact validated v3 xreg prediction as a capped external-feature component alongside the new lora forecast.

In [ ]:
api=wandb.Api();artifact=api.artifact(f"{config['wandb_entity']}/{config['wandb_project']}/{config['v3_artifact']}");artifact_dir=Path(artifact.download(root=str(output_dir/'v3_artifact')))
calibration=pd.read_csv(artifact_dir/'calibration_predictions.csv',parse_dates=['Date']);validation=pd.read_csv(artifact_dir/'validation_predictions.csv',parse_dates=['Date'])
cal_dates=pd.DatetimeIndex(sorted(calibration.Date.unique()));final_dates=pd.DatetimeIndex(sorted(validation.Date.unique()));eval_keys=sorted(map(tuple,validation[['Store','Dept']].drop_duplicates().to_numpy()))
def attach_matrix(frame,dates,matrix,column):
 hmap={d:i for i,d in enumerate(dates)};frame[column]=[float(matrix[key_to_all[(int(r.Store),int(r.Dept))],hmap[r.Date]]) for r in frame.itertuples(index=False)]
attach_matrix(calibration,cal_dates,cal_lora,'TimesFM_LoRA');attach_matrix(validation,final_dates,final_lora,'TimesFM_LoRA')
print({'source_artifact':artifact.name,'calibration_rows':len(calibration),'validation_rows':len(validation),'lora_validation_wmae':wmae(validation.Weekly_Sales,validation.TimesFM_LoRA,validation.IsHoliday)})

## capped corrected blend search

In [ ]:
candidates=['SeasonalNaive52','TimesFM_Raw','TimesFM_Residual','TimesFM_LoRA','TimesFM_XReg'];steps=int(round(1/config['weight_step']));xreg_max=int(round(config['xreg_weight_cap']*steps));rows=[]
def compositions(total,parts,prefix=()):
 if parts==1:yield prefix+(total,);return
 for value in range(total+1):yield from compositions(total-value,parts-1,prefix+(value,))
arrays={name:calibration[name].to_numpy(float) for name in candidates}
for counts in compositions(steps,len(candidates)):
 if counts[-1]>xreg_max:continue
 weights=np.asarray(counts,float)/steps;pred=np.clip(sum(w*arrays[n] for w,n in zip(weights,candidates)),config['clip_min'],config['clip_max'])
 rows.append({**{f'w_{n}':float(w) for n,w in zip(candidates,weights)},'calibration_wmae':wmae(calibration.Weekly_Sales,pred,calibration.IsHoliday)})
weight_search=pd.DataFrame(rows).sort_values('calibration_wmae').reset_index(drop=True);best=weight_search.iloc[0].to_dict()
for frame in [calibration,validation]:frame['TimesFM_v4_Blend']=np.clip(sum(best[f'w_{n}']*frame[n] for n in candidates),config['clip_min'],config['clip_max'])
assert np.isclose(best['calibration_wmae'],wmae(calibration.Weekly_Sales,calibration.TimesFM_v4_Blend,calibration.IsHoliday),atol=1e-9)
print('best weights:',best);display(weight_search.head(10))

In [ ]:
score_names=candidates+['TimesFM_v4_Blend'];scores=[]
for name in score_names:scores.append({'candidate':name,'calibration_wmae':wmae(calibration.Weekly_Sales,calibration[name],calibration.IsHoliday),'validation_wmae':wmae(validation.Weekly_Sales,validation[name],validation.IsHoliday),'validation_mae':float(np.mean(np.abs(validation.Weekly_Sales-validation[name])))})
score_table=pd.DataFrame(scores).sort_values('validation_wmae').reset_index(drop=True);display(score_table)
v4_wmae=float(score_table.loc[score_table.candidate=='TimesFM_v4_Blend','validation_wmae'].iloc[0]);lora_wmae=float(score_table.loc[score_table.candidate=='TimesFM_LoRA','validation_wmae'].iloc[0]);v3_wmae=1588.8029448973086
metrics={'validation/wmae':v4_wmae,'validation/lora_wmae':lora_wmae,'validation/v3_champion_wmae':v3_wmae,'validation/improvement_vs_v3_pct':100*(v3_wmae-v4_wmae)/v3_wmae,'training/best_lora_validation_wmae':best_val,'training/minutes':training_minutes,'training/epochs_completed':len(history_df),'model/trainable_lora_parameters':trainable,'coverage/series':len(eval_keys),'calibration/best_wmae':best['calibration_wmae'],**{f"calibration/{k}":v for k,v in best.items() if k.startswith('w_')}}
prediction_hash=hashlib.sha256(validation.TimesFM_v4_Blend.to_numpy(np.float64).tobytes()).hexdigest();print(metrics);print({'prediction_sha256':prediction_hash})

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,4.5));axes[0].plot(history_df.epoch,history_df.train_wmae,label='train');axes[0].plot(history_df.epoch,history_df.lora_validation_wmae,label='lora validation');axes[0].set_title('lora training');axes[0].legend()
axes[1].bar(score_table.candidate,score_table.validation_wmae);axes[1].set_title('final validation wmae');axes[1].tick_params(axis='x',rotation=30);plt.tight_layout();plot_path=output_dir/'timesfm_v4_diagnostics.png';fig.savefig(plot_path,dpi=160,bbox_inches='tight');plt.show()

## log adapter and evaluation artifacts

In [ ]:
paths={'history':output_dir/'training_history.csv','calibration':output_dir/'calibration_predictions.csv','validation':output_dir/'validation_predictions.csv','weights':output_dir/'weight_search.csv','scores':output_dir/'candidate_scores.csv','metrics':output_dir/'metrics.json'}
history_df.to_csv(paths['history'],index=False);calibration.to_csv(paths['calibration'],index=False);validation.to_csv(paths['validation'],index=False);weight_search.to_csv(paths['weights'],index=False);score_table.to_csv(paths['scores'],index=False)
manifest={'experiment':'timesfm v4 lora + capped xreg hybrid','base_model':config['model_id'],'source_xreg_artifact':artifact.name,'prediction_sha256':prediction_hash,**metrics};paths['metrics'].write_text(json.dumps(manifest,indent=2))
run.log({k:v for k,v in metrics.items() if isinstance(v,(int,float,np.integer,np.floating))});run.log({'results/scores':wandb.Table(dataframe=score_table),'calibration/top_weights':wandb.Table(dataframe=weight_search.head(100)),'validation/predictions':wandb.Table(dataframe=validation.head(20000)),'diagnostics':wandb.Image(str(plot_path))})
adapter_artifact=wandb.Artifact('timesfm-v4-lora-adapter',type='model',description='walmart-adapted timesfm 2.5 lora adapter',metadata=manifest);adapter_artifact.add_dir(str(adapter_dir));run.log_artifact(adapter_artifact,aliases=['v4','latest'])
evaluation_artifact=wandb.Artifact('timesfm-v4-lora-xreg-evaluation',type='evaluation',metadata=manifest)
for p in [*paths.values(),plot_path]:evaluation_artifact.add_file(str(p))
run.log_artifact(evaluation_artifact,aliases=['v4','latest']);run.summary.update(manifest);run.finish()
print({'adapter_artifact':adapter_artifact.name,'evaluation_artifact':evaluation_artifact.name,'output_dir':str(output_dir)})

## decision

v4 becomes the timesfm champion only if its untouched final-validation wmae is below v3's `1588.80`. otherwise the adapter remains a documented experiment and v3 stays champion. model-registry pipeline packaging is intentionally deferred until this comparison is known.